# Kalman Filter with Time-Point Priors and Positive Coefficients

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from wunkui.models.ssp.kalman_1d import (
    kalman_filter_1d,
    kalman_dk_smoother_1d,
    kalman_rts_smoother_1d,
)
from wunkui.models.ssp import plot_states
from wunkui.models.ssp.utils import construct_states_prior

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# False → plain Kalman filter; True → enforce a_obs at sampled windows
USE_DISCLOSURE = True 
# number of disclosure windows to sample
N_PERIODS      = 3    
# consecutive steps per window
N_POINTS       = 10     
# disclose noise variance
DISC_OBS_SIGMA = 0.01

In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
# for quick testing
df = df[-365:].reset_index(drop=True)

saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
response = df["sales"].values

response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)
X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)

Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))
positivity_idx = positivity_idx.astype(bool)

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="y")

In [ ]:
n_steps = y.shape[0]
n_states = Z.shape[-1]
ssp_priors = {}

if coefs_df is not None:
    coefs_lookup = coefs_df.set_index("regressor")["coef"]
    true_states = jnp.array([0.0] + [coefs_lookup[r] for r in regressors])

    if USE_DISCLOSURE:
        ssp_priors = construct_states_prior(
            n_steps=n_steps,
            n_states=n_states,
            true_states=true_states,
            regressors=regressors,
            n_periods=N_PERIODS,
            n_points=N_POINTS,
            obs_scale=DISC_OBS_SIGMA,
        )
        print("a_obs shape:", ssp_priors["a_obs"].shape)
        print("P_obs shape:", ssp_priors["P_obs"].shape)
        print("n disclosed steps:", len(ssp_priors["obs_idx"]))
    else:
        ssp_priors.update({"a_obs": None, "P_obs": None, "obs_idx": np.array([], dtype=int)})
        print("disclosure disabled")
else:
    ssp_priors.update({"a_obs": None, "P_obs": None, "obs_idx": np.array([], dtype=int)})
    print("no ground truth state available; disclosure disabled")

In [ ]:
ssp_priors["a0"] = jnp.array([0.0] + [0.1] * (X.shape[-1]))
ssp_priors["P0"] = jnp.array([1.0] + [0.1] * (X.shape[-1]))
# (level, regressor) local scale priors
ssp_priors["sigma_q_loc_prior"] = np.array([0.05, 0.01])
ssp_priors["sigma_q_scale_prior"] = np.array([0.05, 0.01])

In [ ]:
# Test run to verify dimensions are correct before sampling
def test_run(ssp_priors, positivity_idx):
    a0 = ssp_priors["a0"]
    P0 = ssp_priors["P0"]
    a_obs = ssp_priors["a_obs"]
    P_obs = ssp_priors["P_obs"]

    sigma_h = jnp.array(0.1)
    sigma_q_raw = ssp_priors["sigma_q_loc_prior"]  # using prior loc as a test value
    sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
    print("sigma_q expanded:", sigma_q.shape, sigma_q)

    lp, at, Pt, _, _, _ = kalman_filter_1d(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y,
        Z=Z,
        logp=True,
        a_obs=a_obs,
        P_obs=P_obs,
        positivity_idx=positivity_idx,
    )
    print("lp:", lp)
    print("at shape:", at.shape)
    print("Pt shape:", Pt.shape)

test_run(ssp_priors, positivity_idx)

In [ ]:
def make_nuts_fn(ssp_priors, y, Z, positivity_idx):
    a0                  = ssp_priors["a0"]
    P0                  = ssp_priors["P0"]
    sigma_q_loc_prior   = ssp_priors["sigma_q_loc_prior"]
    sigma_q_scale_prior = ssp_priors["sigma_q_scale_prior"]
    a_obs               = ssp_priors["a_obs"]
    P_obs               = ssp_priors["P_obs"]

    def nuts_fn():
        sigma_h = numpyro.sample(
            "sigma_h",
            dist.TruncatedNormal(0.1, 1.0, high=1.0, low=1e-5),
        )

        sigma_q_raw = numpyro.sample(
            "sigma_q",
            dist.TruncatedNormal(sigma_q_loc_prior, sigma_q_scale_prior, high=0.1, low=1e-5),
        )

        n_regressors = Z.shape[-1] - 1
        sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], n_regressors)])

        lp, at, Pt, _, _, _ = kalman_filter_1d(
            a0=a0,
            P0=P0,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y,
            Z=Z,
            logp=True,
            a_obs=a_obs,
            P_obs=P_obs,
            positivity_idx=positivity_idx,
        )
        numpyro.factor("lp", lp)

        # Stash filter outputs and the expanded sigma_q so the RTS smoother
        # can be vmapped over posterior draws as a pure-JAX post-processing
        # step (see cell after `mcmc.get_samples`).
        numpyro.deterministic("at", at)
        numpyro.deterministic("Pt", Pt)
        numpyro.deterministic("sigma_q_full", sigma_q)

    return nuts_fn

In [ ]:
nuts_fn = make_nuts_fn(ssp_priors, y, Z, positivity_idx)
nuts_kernel = NUTS(nuts_fn)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=400,
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key)

In [ ]:
from numpyro.diagnostics import summary

# print_summary covers R-hat and n_eff for all sites
mcmc.print_summary()

# detailed per-parameter diagnostics for scalar params
samples = mcmc.get_samples(group_by_chain=True)
diag = summary(samples, prob=0.9)

# flag any R-hat > 1.01 or n_eff < 100
# use max(r_hat) / min(n_eff) for multi-dimensional params (at, lam, mu, etc.)
print("\n--- Convergence flags ---")
for param, stats in diag.items():
    rhat = float(np.asarray(stats["r_hat"]).max())
    neff = float(np.asarray(stats["n_eff"]).min())
    if rhat > 1.01 or neff < 100:
        print(f"  WARNING  {param:<20}  r_hat={rhat:.4f}  n_eff={neff:.1f}")

In [ ]:
posteriors_dict = mcmc.get_samples()
posteriors_dict.keys()

In [ ]:
# ── Post-MCMC RTS smoother ────────────────────────────────────────────────
# The filter ran inside the NumPyro model and stashed (at, Pt, sigma_q_full)
# as deterministic sites.  RTS is a pure-JAX backward pass, so we vmap it
# over posterior draws here rather than re-running it inside the trace.

at_draws       = posteriors_dict["at"]            # (n_draws, T, n_states)
Pt_draws       = posteriors_dict["Pt"]            # (n_draws, T, n_states)
sigma_q_draws  = posteriors_dict["sigma_q_full"]  # (n_draws, n_states)

at_smooth, Pt_smooth = jax.vmap(kalman_rts_smoother_1d)(
    at_draws, Pt_draws, sigma_q_draws,
)

# ── Verification: parity vs D&K + variance sanity ────────────────────────
# 1) On a single draw, RTS smoothed means must match the D&K smoother that
#    used to live inside the model.  D&K needs vt/Ft/Kt, which aren't stored
#    post-MCMC, so re-run the filter for one draw to grab them.
_idx = 0
_lp, _at_f, _Pt_f, _vt, _Ft, _Kt = kalman_filter_1d(
    a0=ssp_priors["a0"],
    P0=ssp_priors["P0"],
    sigma_h=posteriors_dict["sigma_h"][_idx],
    sigma_q=sigma_q_draws[_idx],
    y=y,
    Z=Z,
    logp=False,
    a_obs=ssp_priors["a_obs"],
    P_obs=ssp_priors["P_obs"],
    positivity_idx=positivity_idx,
)
_at_dk = kalman_dk_smoother_1d(
    _at_f, _Pt_f, _vt, _Ft, _Kt, Z,
    ssp_priors["a0"], ssp_priors["P0"],
    ssp_priors["a_obs"], ssp_priors["P_obs"],
)
_at_rts, _Pt_rts = kalman_rts_smoother_1d(_at_f, _Pt_f, sigma_q_draws[_idx])
_mean_diff = float(jnp.max(jnp.abs(_at_rts - _at_dk)))
print(f"max |RTS mean − D&K mean| on draw {_idx}: {_mean_diff:.2e}")

# 2) Variance sanity: at t=T-1 the smoothed variance equals the filter
#    posterior variance Pt[-1] - sigma_q^2; for t<T-1 it cannot exceed it.
_P_filt = _Pt_f - jnp.square(sigma_q_draws[_idx])
_boundary_diff = float(jnp.max(jnp.abs(_Pt_rts[-1] - _P_filt[-1])))
_violations = int(jnp.sum(_Pt_rts[:-1] > _P_filt[:-1] + 1e-9))
print(f"|Pt_smooth[-1] − P_filt[-1]| = {_boundary_diff:.2e}")
print(f"smoothing-reduces-variance violations (should be 0): {_violations}")

# 3) Vmap shape check
print(f"at_smooth shape: {at_smooth.shape}    Pt_smooth shape: {Pt_smooth.shape}")

# ── Inject smoothed quantities back into the posterior dict ──────────────
# `mu` keeps its old name so the forecast cell below works unchanged.  We
# additionally expose the within-draw smoothing variance via a draw of total
# uncertainty: mean + N(0, P_{t|T}).  Plotting `at_smooth_total` with the
# existing plot_states gives credible bands that include *both* across-draw
# and within-draw variance via the law of total variance.
_rng = random.PRNGKey(7)
_eps = random.normal(_rng, at_smooth.shape) * jnp.sqrt(jnp.clip(Pt_smooth, a_min=0.0))
posteriors_dict["at_smooth"]       = np.asarray(at_smooth)
posteriors_dict["Pt_smooth"]       = np.asarray(Pt_smooth)
posteriors_dict["at_smooth_total"] = np.asarray(at_smooth + _eps)
posteriors_dict["mu"]              = np.asarray(jnp.einsum("dts,ts->dt", at_smooth, Z))

In [ ]:
state_labels = ["intercept"] + regressors
a_obs = ssp_priors["a_obs"] if USE_DISCLOSURE else None
P_obs = ssp_priors["P_obs"] if USE_DISCLOSURE else None
obs_idx = ssp_priors["obs_idx"] if USE_DISCLOSURE else None

plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="at",
    coefs_df=coefs_df,
    obs_idx=obs_idx,
    a_obs=a_obs,
    P_obs=P_obs,
    title="Filtered state coefficients — disclosure ON" if USE_DISCLOSURE else "Filtered state coefficients — disclosure OFF",
)
plt.show()

# Across-draw bands of the smoothed mean (E_θ[a_{t|T}]).  Comparable to the
# pre-RTS plot — bands reflect MCMC uncertainty in σ_h, σ_q only.
plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="at_smooth",
    coefs_df=coefs_df,
    obs_idx=obs_idx,
    a_obs=a_obs,
    P_obs=P_obs,
    title="Smoothed state coefficients (across-draw only) — disclosure ON" if USE_DISCLOSURE else "Smoothed state coefficients (across-draw only) — disclosure OFF",
)
plt.show()

# Total smoothed uncertainty: E_θ[a_{t|T}] + Var_θ[a_{t|T}] ⊕ E_θ[P_{t|T}]
# realised by drawing one Gaussian sample per posterior draw using the RTS
# variance (`at_smooth_total` injected above).  Bands here include both
# across-draw spread and the within-draw smoothing variance — the new
# capability unlocked by RTS.
plot_states(
    posteriors_dict,
    df["date"].values,
    state_labels,
    states_key="at_smooth_total",
    coefs_df=coefs_df,
    obs_idx=obs_idx,
    a_obs=a_obs,
    P_obs=P_obs,
    title="Smoothed state coefficients (total variance) — disclosure ON" if USE_DISCLOSURE else "Smoothed state coefficients (total variance) — disclosure OFF",
)
plt.show()

In [ ]:
mu_samples = np.array(posteriors_dict["mu"])
eps_samples = np.random.normal(
    0,
    np.array(posteriors_dict["sigma_h"])[:, None],
    size=mu_samples.shape,
)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)

n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')